# 鼠标拖动版华容道游戏

本 Notebook 实现一个可以通过鼠标拖动颜色块进行游玩的华容道小游戏。

功能特点：

1. 使用经典 `4 x 5` 华容道棋盘。
2. 不同角色使用不同颜色块表示。
3. 按住棋子并拖动，松开鼠标后尝试移动棋子。
4. 如果移动合法，棋子会吸附到新格子；如果不合法，棋子会回到原位置。
5. 当曹操移动到棋盘底部出口位置时，游戏胜利。

运行提示：

- 鼠标拖动交互需要 Jupyter 支持 Matplotlib 交互后端。
- 如果使用 Jupyter Notebook / JupyterLab，建议先安装 `ipympl`：

```bash
pip install ipympl
```

或：

```bash
conda install -c conda-forge ipympl
```


## 1. 游戏规则与状态表示

棋盘大小为 `4 x 5`。

棋子包括：

- `CaoCao`：`2 x 2` 大方块，目标是移动到底部出口。
- `GuanYu`：`2 x 1` 横向长方块。
- `ZhangFei`、`ZhaoYun`、`MaChao`、`HuangZhong`：`1 x 2` 纵向长方块。
- `Soldier`：四个 `1 x 1` 小方块。

状态使用每个棋子的左上角坐标 `(row, col)` 表示。


## 2. 鼠标拖动版华容道程序

运行下面代码单元即可启动游戏。

操作方式：

1. 用鼠标按住一个颜色块。
2. 向上、下、左、右拖动。
3. 松开鼠标。
4. 如果移动合法，棋子会移动一格；如果不合法，会自动回到原位置。

说明：为了符合华容道规则，每次松开鼠标后最多移动一格。


In [ ]:

from dataclasses import dataclass

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

try:
    from IPython.display import display
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    widgets = None
    HAS_WIDGETS = False

# 鼠标拖动需要交互式 Matplotlib 后端。
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    pass


BOARD_WIDTH = 4
BOARD_HEIGHT = 5
GOAL_CAOCAO_POSITION = (3, 1)


@dataclass(frozen=True)
class Piece:
    """
    华容道棋子定义。

    属性：
        name: 棋子名称。
        width: 棋子宽度，占用几列。
        height: 棋子高度，占用几行。
        color: 棋子颜色。
        label: 棋子显示文字。
    """
    name: str
    width: int
    height: int
    color: str
    label: str


PIECES = (
    Piece('CaoCao', 2, 2, '#d94f45', 'CaoCao'),
    Piece('ZhangFei', 1, 2, '#7b4ab2', 'Zhang'),
    Piece('ZhaoYun', 1, 2, '#3b82c4', 'Zhao'),
    Piece('MaChao', 1, 2, '#2f9e44', 'Ma'),
    Piece('HuangZhong', 1, 2, '#8f6b32', 'Huang'),
    Piece('GuanYu', 2, 1, '#e0a526', 'Guan'),
    Piece('Soldier1', 1, 1, '#8d99ae', 'S1'),
    Piece('Soldier2', 1, 1, '#8d99ae', 'S2'),
    Piece('Soldier3', 1, 1, '#8d99ae', 'S3'),
    Piece('Soldier4', 1, 1, '#8d99ae', 'S4'),
)

INITIAL_STATE = (
    (0, 1),  # CaoCao
    (0, 0),  # ZhangFei
    (0, 3),  # ZhaoYun
    (2, 0),  # MaChao
    (2, 3),  # HuangZhong
    (2, 1),  # GuanYu
    (3, 1),  # Soldier1
    (3, 2),  # Soldier2
    (4, 0),  # Soldier3
    (4, 3),  # Soldier4
)


def build_board(state):
    """
    根据当前状态生成棋盘占用表。

    参数：
        state: 每个棋子的左上角坐标。

    返回：
        board: 二维列表。空格为 None，被棋子占用的位置保存棋子编号。
    """
    board = [[None for _ in range(BOARD_WIDTH)] for _ in range(BOARD_HEIGHT)]

    for piece_index, (row, col) in enumerate(state):
        piece = PIECES[piece_index]
        for dr in range(piece.height):
            for dc in range(piece.width):
                r = row + dr
                c = col + dc
                if r < 0 or r >= BOARD_HEIGHT or c < 0 or c >= BOARD_WIDTH:
                    return None
                if board[r][c] is not None:
                    return None
                board[r][c] = piece_index

    return board


def can_place_piece(board, piece_index, new_row, new_col):
    """
    判断棋子能否放置到指定位置。

    参数：
        board: 当前棋盘占用表。
        piece_index: 棋子编号。
        new_row: 新位置左上角行号。
        new_col: 新位置左上角列号。

    返回：
        True 表示合法；False 表示非法。
    """
    piece = PIECES[piece_index]

    if new_row < 0 or new_col < 0:
        return False
    if new_row + piece.height > BOARD_HEIGHT or new_col + piece.width > BOARD_WIDTH:
        return False

    for dr in range(piece.height):
        for dc in range(piece.width):
            occupied_by = board[new_row + dr][new_col + dc]
            if occupied_by is not None and occupied_by != piece_index:
                return False

    return True


def is_goal(state):
    """
    判断曹操是否到达出口位置。
    """
    return state[0] == GOAL_CAOCAO_POSITION


class DragHuarongDaoGame:
    """
    鼠标拖动版华容道游戏。

    核心思路：
        1. 鼠标按下时，根据点击坐标选中棋子。
        2. 鼠标移动时，临时拖动该棋子的矩形块。
        3. 鼠标松开时，根据拖动方向尝试移动一格。
        4. 如果合法，更新状态；如果非法，恢复原位。
    """

    def __init__(self, start_state):
        self.start_state = tuple(start_state)
        self.state = tuple(start_state)
        self.step_count = 0
        self.selected_piece = None
        self.drag_start_mouse = None
        self.drag_start_position = None
        self.rectangles = {}
        self.labels = {}
        self.message = 'Drag a block to play.'

        self.fig = None
        self.ax = None
        self.status_widget = None
        self.canvas_displayed = False

    def reset(self):
        """
        重置游戏。
        """
        self.state = self.start_state
        self.step_count = 0
        self.selected_piece = None
        self.message = 'Game reset. Drag a block to play.'
        self.draw()

    def piece_at(self, x, y):
        """
        根据鼠标坐标判断点击的是哪个棋子。

        参数：
            x: 鼠标所在列方向坐标。
            y: 鼠标所在行方向坐标。

        返回：
            piece_index 或 None。
        """
        if x is None or y is None:
            return None

        col = int(x)
        row = int(y)

        if row < 0 or row >= BOARD_HEIGHT or col < 0 or col >= BOARD_WIDTH:
            return None

        board = build_board(self.state)
        return board[row][col]

    def move_piece(self, piece_index, direction):
        """
        按指定方向移动棋子一格。

        参数：
            piece_index: 棋子编号。
            direction: 移动方向，取值为 Up、Down、Left、Right。

        返回：
            True 表示移动成功；False 表示移动失败。
        """
        direction_map = {
            'Up': (-1, 0),
            'Down': (1, 0),
            'Left': (0, -1),
            'Right': (0, 1),
        }
        dr, dc = direction_map[direction]
        row, col = self.state[piece_index]
        new_row = row + dr
        new_col = col + dc

        board = build_board(self.state)
        if not can_place_piece(board, piece_index, new_row, new_col):
            self.message = f'Illegal move: {PIECES[piece_index].name} {direction}'
            return False

        next_state = list(self.state)
        next_state[piece_index] = (new_row, new_col)
        self.state = tuple(next_state)
        self.step_count += 1
        self.message = f'Move {self.step_count}: {PIECES[piece_index].name} {direction}'

        if is_goal(self.state):
            self.message += ' | Success! CaoCao reached the exit.'

        return True

    def decide_direction(self, start_mouse, end_mouse):
        """
        根据拖动起点和终点判断移动方向。

        参数：
            start_mouse: 鼠标按下时的坐标。
            end_mouse: 鼠标松开时的坐标。

        返回：
            方向字符串，或 None。
        """
        start_x, start_y = start_mouse
        end_x, end_y = end_mouse
        dx = end_x - start_x
        dy = end_y - start_y

        if abs(dx) < 0.25 and abs(dy) < 0.25:
            return None

        if abs(dx) > abs(dy):
            return 'Right' if dx > 0 else 'Left'
        return 'Down' if dy > 0 else 'Up'

    def draw(self):
        """
        绘制游戏棋盘。
        """
        if self.ax is None:
            self.fig, self.ax = plt.subplots(figsize=(4.8, 6.0))
            self.fig.canvas.mpl_connect('button_press_event', self.on_press)
            self.fig.canvas.mpl_connect('motion_notify_event', self.on_motion)
            self.fig.canvas.mpl_connect('button_release_event', self.on_release)
        else:
            self.ax.clear()

        ax = self.ax
        ax.set_xlim(0, BOARD_WIDTH)
        ax.set_ylim(0, BOARD_HEIGHT)
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_facecolor('#f7f2e8')
        ax.set_title(f'Huarong Dao Drag Game | Steps: {self.step_count}')

        for x in range(BOARD_WIDTH + 1):
            ax.plot([x, x], [0, BOARD_HEIGHT], color='#5c4a36', linewidth=1)
        for y in range(BOARD_HEIGHT + 1):
            ax.plot([0, BOARD_WIDTH], [y, y], color='#5c4a36', linewidth=1)

        ax.plot([1, 3], [5, 5], color='#b00020', linewidth=5, solid_capstyle='round')
        ax.text(2, 4.82, 'EXIT', ha='center', va='center', color='#b00020', fontsize=10, weight='bold')

        self.rectangles = {}
        self.labels = {}

        for piece_index, (row, col) in enumerate(self.state):
            piece = PIECES[piece_index]
            edge_color = 'black'
            edge_width = 3 if piece_index == self.selected_piece else 1.8

            rect = Rectangle(
                (col, row),
                piece.width,
                piece.height,
                facecolor=piece.color,
                edgecolor=edge_color,
                linewidth=edge_width,
                alpha=0.92,
            )
            ax.add_patch(rect)
            label = ax.text(
                col + piece.width / 2,
                row + piece.height / 2,
                piece.label,
                ha='center',
                va='center',
                fontsize=10,
                weight='bold',
                color='white',
            )

            self.rectangles[piece_index] = rect
            self.labels[piece_index] = label

        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

        if self.status_widget is not None:
            self.status_widget.value = self.message

        self.fig.canvas.draw_idle()

    def on_press(self, event):
        """
        鼠标按下事件：选择棋子并记录拖动起点。
        """
        if event.inaxes != self.ax:
            return

        piece_index = self.piece_at(event.xdata, event.ydata)
        if piece_index is None:
            self.selected_piece = None
            self.message = 'Please drag a colored block.'
            self.draw()
            return

        self.selected_piece = piece_index
        self.drag_start_mouse = (event.xdata, event.ydata)
        self.drag_start_position = self.state[piece_index]
        self.message = f'Selected: {PIECES[piece_index].name}'
        self.draw()

    def on_motion(self, event):
        """
        鼠标移动事件：临时拖动显示。
        """
        if self.selected_piece is None or self.drag_start_mouse is None:
            return
        if event.inaxes != self.ax or event.xdata is None or event.ydata is None:
            return

        start_x, start_y = self.drag_start_mouse
        start_row, start_col = self.drag_start_position
        dx = event.xdata - start_x
        dy = event.ydata - start_y

        rect = self.rectangles[self.selected_piece]
        label = self.labels[self.selected_piece]
        piece = PIECES[self.selected_piece]

        rect.set_xy((start_col + dx, start_row + dy))
        label.set_position((start_col + dx + piece.width / 2, start_row + dy + piece.height / 2))
        self.fig.canvas.draw_idle()

    def on_release(self, event):
        """
        鼠标松开事件：判断拖动方向并尝试移动。
        """
        if self.selected_piece is None or self.drag_start_mouse is None:
            return

        if event.xdata is None or event.ydata is None:
            self.message = 'Move cancelled.'
            self.drag_start_mouse = None
            self.drag_start_position = None
            self.draw()
            return

        direction = self.decide_direction(self.drag_start_mouse, (event.xdata, event.ydata))
        piece_index = self.selected_piece

        if direction is None:
            self.message = 'Drag farther to move a block.'
        else:
            self.move_piece(piece_index, direction)

        self.drag_start_mouse = None
        self.drag_start_position = None
        self.draw()

    def show(self):
        """
        显示游戏。

        注意：画布只 display 一次。之后拖动棋子时只更新同一个画布，
        不再生成新的结果图。
        """
        self.draw()

        if HAS_WIDGETS:
            reset_button = widgets.Button(description='Reset')

            def on_reset(_):
                self.reset()

            reset_button.on_click(on_reset)
            display(reset_button)

            if not self.canvas_displayed:
                display(self.fig.canvas)
                self.canvas_displayed = True
        else:
            if not self.canvas_displayed:
                plt.show()
                self.canvas_displayed = True


game = DragHuarongDaoGame(INITIAL_STATE)
game.show()
